# Qubitization Phase Estimation
Adapted from Qualtran v0.7.0. Copyright 2024 Google LLC, Apache-2.0. Frozen commit `bcb17bb25360ed68ce3bc084645db3e929799389`.

All thirteen original cells are mapped. The four large factory examples are typed declarations and one-level call graphs. Executable composition uses `circuit.append(op)` as verified in `QUBITIZATION_PHASE_ESTIMATION_FILE_AUDIT.md`. Binding these declarations requires explicit admitted physical children and window PREPARE.


In [ ]:
from collections.abc import Callable, Mapping
from typing import cast

import numpy as np
from IPython import display as notebook_display

from examples.algorithms.hubbard_qubitization_resources import notebook_signature
from examples.algorithms.sparse_chemistry_resources import sparse_integral_fixture
from flagquantum.algorithms.primitives.hubbard_walk import HubbardWalk
from flagquantum.algorithms.primitives.ising_walk import ising_walk
from flagquantum.algorithms.primitives.luis_perina_state import LuisPerinaState
from flagquantum.algorithms.primitives.multiplexers.thc_resource_model import (
    select_thc_signature,
)
from flagquantum.algorithms.primitives.phase_estimation_window import (
    PhaseEstimationWindow,
    RectangularWindowState,
)
from flagquantum.algorithms.primitives.qubitization_phase_estimation import (
    QubitizationPhaseEstimation,
)
from flagquantum.algorithms.primitives.sparse_chemistry_walk import SparseChemistryWalk
from flagquantum.algorithms.primitives.thc_chemistry_data import thc_chemistry_data
from flagquantum.algorithms.primitives.thc_chemistry_prepare import PrepareTHCChemistry
from flagquantum.core.ir import Instruction
from flagquantum.core.ir.program.register_types import (
    bounded_unsigned_register_type,
    opaque_register_type,
)
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.signature_graph import export_signature_graph_dot

SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


def unresolved_walk(
    ports: Mapping[str, tuple[int, ...]],
    workspace: tuple[int, ...],
    classical_bits: tuple[int, ...],
    adjoint: bool,
) -> tuple[Instruction, ...]:
    raise NotImplementedError(
        "These chemistry factory examples declare ports and costs. Physical QPE child admission remains open in the file audit."
    )


def unresolved_controlled_walk(
    control: int,
    ports: Mapping[str, tuple[int, ...]],
    workspace: tuple[int, ...],
    classical_bits: tuple[int, ...],
    adjoint: bool,
) -> tuple[Instruction, ...]:
    return unresolved_walk(ports, workspace, classical_bits, adjoint)


def declaration(
    walk_signature: Signature,
    reflection_signature: Signature,
    window: PhaseEstimationWindow,
) -> QubitizationPhaseEstimation:
    return QubitizationPhaseEstimation(
        walk_signature,
        reflection_signature,
        window,
        unresolved_walk,
        unresolved_controlled_walk,
        unresolved_controlled_walk,
    )


def show_signature(name: str, signature: Signature) -> None:
    display(SVG(render_call_graph_svg(export_signature_graph_dot(signature, name))))


## `QubitizationPhaseEstimation`
Heisenberg limited phase estimation circuit for learning eigenphase of `walk`.

The Bloq yields an OPTREE to construct Heisenberg limited phase estimation circuit
for learning eigenphases of the `walk` operator with `m` bits of accuracy. The
circuit is implemented as given in Fig.2 of Ref-1.

```
       ┌─────────┐                                     ┌─────────┐
  |0> -│         │-------------------------(0)---(0)---│         │---M--- [m1]:highest bit
       │         │                          |     |    │         │
  |0> -│         │----------------(0)---(0)-+-----+----│         │---M--- [m2]
       │CtrlState│                 |     |  |     |    │ QFT_inv │
  |0> -│  Prep   │-------(0)---(0)-+-----+--+-----+----│         │---M--- [m3]
       │         │        |     |  |     |  |     |    │         │
  |0> -│         │---@----+-----+--+-----+--+-----+----│         │---M--- [m4]:lowest bit
       └─────────┘   |    |     |  |     |  |     |    └─────────┘
|Psi> ---------------W----R-W^2-R--R-W^4-R--R-W^8-R---------------------- |Psi>

```

TODO: Note that there are slight differences between the Fig2 of the Ref[1] and the circuit
      implemented here. Further investigation is required to reconcile the difference.

#### Parameters
 - `walk`: Bloq representing the Qubitization walk operator to run the phase estimation protocol on.
 - `ctrl_state_prep`: Bloq to prepare the control state on the phase register. Defaults to `OnEach(self.m_bits, Hadamard())`.
 - `qft_inv`: Bloq to apply inverse QFT on the phase register. Defaults to `QFTTextBook(self.m_bits).adjoint()`  

#### Registers
 - `qpe_reg`: Control register of type `QFxp(self.m_bits, self.m_bits)` for phase estimation.
 - `target registers`: All registers used in `self.unitary.signature` 

#### References
 - [Encoding Electronic Spectra in Quantum Circuits with Linear T Complexity](https://arxiv.org/abs/1805.03662). Fig. 2


In [ ]:
# The shared QubitizationPhaseEstimation composition owns the frozen schedule.


### Example Instances

In [ ]:
x_dim, y_dim, t = 2, 2, 2
u = 4 * t
hubbard_walk = HubbardWalk.from_lattice(x_dim, t, u)
algo_eps = t / 100
N = x_dim * y_dim * 2
qlambda = 2 * N * t + (N * u) // 2
qpe_eps = algo_eps / (qlambda * np.sqrt(2))
hubbard_signature = notebook_signature(x_dim)
qubitization_qpe_hubbard_model_small = declaration(
    hubbard_signature,
    Signature(tuple(hubbard_signature)[:-1]),
    LuisPerinaState.from_standard_deviation_eps(qpe_eps),
)


In [ ]:
num_spatial = 6
tpq, eris = sparse_integral_fixture(num_spatial // 2)
sparse_walk = SparseChemistryWalk.from_integrals(
    tpq, eris, probability_bits=16, rotation_bits=8
)
algo_eps = 0.0016
qlambda = np.sum(np.abs(tpq)) + 0.5 * np.sum(np.abs(eris))
qpe_eps = algo_eps / (qlambda * np.sqrt(2))
qubitization_qpe_sparse_chem = declaration(
    sparse_walk.source_signature,
    Signature(sparse_walk.prepare.selection_registers),
    LuisPerinaState.from_standard_deviation_eps(qpe_eps),
)


In [ ]:
# Preserve the original seeded 450-factor, 152-spin-orbital declaration.
num_spinorb, num_bits_state_prep, num_bits_rot, thc_dim = 152, 10, 20, 450
num_spat = num_spinorb // 2
# The inverse-QROM optimum considers the adjacent integer exponents.
length = thc_dim + num_spat
exponent = 0.5 * np.log2(length)
candidates = (int(np.floor(exponent)), int(np.ceil(exponent)))
qroam_blocking_factor = 2 ** min(candidates, key=lambda k: length / 2**k + 2**k)
rng = np.random.RandomState(7)
one = rng.normal(size=(num_spat, num_spat))
t_l = np.linalg.eigvalsh(0.5 * (one + one.T))
central = rng.normal(size=(thc_dim, thc_dim))
zeta = 0.5 * (central + central.T)
eta = rng.normal(size=(thc_dim, num_spat))
thc_data = thc_chemistry_data(t_l, eta, zeta, probability_bits=num_bits_state_prep)
thc_prepare = PrepareTHCChemistry(thc_data)
thc_signature = select_thc_signature(thc_dim, num_spinorb, num_bits_state_prep)
algo_eps = 0.0016
qpe_eps = algo_eps / (thc_data.normalization * 2**0.5)
qubitization_qpe_chem_thc = declaration(
    thc_signature,
    Signature(thc_prepare.selection_registers),
    LuisPerinaState.from_standard_deviation_eps(qpe_eps),
)


In [ ]:
ising, _ = ising_walk(4, 0.1)
ising_selection = Register(
    "selection",
    bounded_unsigned_register_type(
        ising.selector.selection_bitsize, len(ising.table.keep)
    ),
)
ising_signature = Signature(
    (ising_selection, Register("target", opaque_register_type(4)))
)
qubitization_qpe_ising = declaration(
    ising_signature, Signature((ising_selection,)), RectangularWindowState(4)
)


#### Graphical Signature

In [ ]:
for name, operation in (
    ("qubitization_qpe_hubbard_model_small", qubitization_qpe_hubbard_model_small),
    ("qubitization_qpe_sparse_chem", qubitization_qpe_sparse_chem),
    ("qubitization_qpe_chem_thc", qubitization_qpe_chem_thc),
    ("qubitization_qpe_ising", qubitization_qpe_ising),
):
    show_signature(name, operation.signature)


### Call Graph

In [ ]:
qubitization_qpe_hubbard_model_small_graph = (
    qubitization_qpe_hubbard_model_small.source_graph(
        root="QubitizationPhaseEstimation",
        preparation="LuisPerinaState",
        controlled_walk="ControlledWalk(1)",
        zero_controlled_reflection="Reflection(0)",
        walk="Walk",
        inverse_qft="QFT†",
    )
)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(
                qubitization_qpe_hubbard_model_small_graph.root,
                qubitization_qpe_hubbard_model_small_graph.direct_calls,
            )
        )
    )
)
display(qubitization_qpe_hubbard_model_small_graph.leaf_counts)


## Execution and separate resource accounts

The source notebook only constructs declarations and graphs. Its unresolved paper Fig. 2 discrepancy above is retained. Q-0631 tracks large factory physical admission and SDK-default leaf synthesis. Source set-union port order is nondeterministic, so declaration comparisons preserve complete descriptors by name.

The accepted two-bit reference-child composition emits 105 primitive instructions, including one measurement, with one work qubit and one feedback slot. Its independent declared model is 2 And, 45 Clifford and 2 measurements, with peak 5 qubits. The LP variant emits 227 primitives and 4 measurements with 6 work qubits and 3 slots. Its model is 5 And, 92 Clifford, 18 rotations and 5 measurements, with peak 10. These measured small fixtures are documented in the source-file audit. The large declarations above have no claimed primitive inventory or simulated amplitudes.
